# Step 9 — Barrier Options

A **barrier option** is a path-dependent contract whose payoff depends on whether the underlying asset crosses a pre-specified barrier level $H$ during the life of the option.

| Type | Description |
|------|-------------|
| **Down-and-out** | Expires worthless if $S$ falls to $H < S_0$ |
| **Down-and-in**  | Pays off *only if* $S$ falls to $H < S_0$ |
| **Up-and-out**   | Expires worthless if $S$ rises to $H > S_0$ |
| **Up-and-in**    | Pays off *only if* $S$ rises to $H > S_0$ |

This notebook focuses on the **down-and-out call**: a standard European call that is knocked out (set to zero) if the stock ever touches $H < S_0$. This is the most common barrier variant and cleanly illustrates the finite-difference treatment.

## Key insight: the barrier is a boundary condition

For a knock-out option the barrier $H$ is simply a **second lower boundary** in the PDE grid:

$$V(H,\, t) = 0 \qquad \forall\; t \in [0, T]$$

Rather than running the grid from $S = 0$ to $S_\text{max}$, we restrict it to $[H,\, S_\text{max}]$ and enforce the knock-out condition at the lower edge. The Crank–Nicolson scheme inside the domain is unchanged — only the domain and one boundary value differ from the vanilla solver.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded
from scipy.stats import norm

from finite_difference import bs_price, solve_crank_nicolson

## Parameters

In [ ]:
K     = 100.0
r     = 0.05
sigma = 0.25
T     = 1.0
S0    = 100.0
H     = 85.0    # barrier level (below spot and strike)
S_max = 300.0
M, N  = 200, 200

vanilla_call = bs_price(S0, K, T, r, sigma, option_type='call')
print(f'Vanilla call (Black–Scholes): {vanilla_call:.4f}')

## 1. Merton (1973) Closed Form

For a down-and-out call with $H \leq K$, Merton (1973) showed that the price is given by the vanilla Black–Scholes price minus an "image" correction arising from the reflection principle for geometric Brownian motion:

$$C_{\text{do}}(S) = C_{\text{BS}}(S) - \left(\frac{H}{S}\right)^{2\lambda} C_{\text{BS}}\!\left(\frac{H^2}{S}\right)$$

where

$$\lambda = \frac{2\left(r - \tfrac{1}{2}\sigma^2\right)}{\sigma^2} = \frac{2r}{\sigma^2} - 1$$

and $C_{\text{BS}}(x)$ is the standard Black–Scholes call price evaluated at spot $x$ (with all other parameters fixed).

**Intuition:** The correction term prices a hypothetical "image" option with spot $H^2/S$ — the reflection of $\ln S$ about $\ln H$ in log-space — and subtracts it with a weight $(H/S)^{2\lambda}$ derived from the Girsanov change of measure. When $S \to H$ the two terms cancel and $C_{\text{do}} \to 0$, enforcing the knock-out. When $H \to 0$ the correction vanishes and $C_{\text{do}} \to C_{\text{BS}}$.

In [ ]:
def bs_call(S, K, T, r, sigma):
    """Standard Black–Scholes call price."""
    if S <= 0 or T <= 0:
        return max(S - K, 0.0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def down_and_out_call_analytical(S, K, T, r, sigma, H):
    """Merton (1973) closed form for a down-and-out call. Requires H <= K."""
    if H > K:
        raise ValueError('Formula requires H <= K')
    if S <= H:
        return 0.0  # already knocked out
    lam    = 2.0 * r / sigma**2 - 1.0           # = 2*(r - sigma^2/2) / sigma^2
    c_van  = bs_call(S, K, T, r, sigma)
    c_img  = bs_call(H**2 / S, K, T, r, sigma)  # image source at H^2/S
    return c_van - (H / S)**lam * c_img


analytical = down_and_out_call_analytical(S0, K, T, r, sigma, H)
print(f'Down-and-out call (Merton):  {analytical:.4f}')
print(f'Vanilla call (BS):           {vanilla_call:.4f}')
print(f'Barrier discount:            {vanilla_call - analytical:.4f}')

## 2. Finite-Difference Solver

The FD approach modifies the vanilla Crank–Nicolson solver in exactly two ways:

1. **Domain**: the stock grid runs from $H$ to $S_\text{max}$ (not $0$ to $S_\text{max}$), so $\Delta S = (S_\text{max} - H)/M$.
2. **Lower boundary**: $V(H, t) = 0$ for all $t$ (knocked-out condition).

Because the grid no longer starts at zero, the coefficients must use actual stock prices $S_i = H + i\,\Delta S$ rather than integer indices $i$:

$$a_i = \frac{\Delta t}{4}\left(\frac{\sigma^2 S_i^2}{\Delta S^2} - \frac{r S_i}{\Delta S}\right), \quad
  b_i = -\frac{\Delta t}{2}\left(\frac{\sigma^2 S_i^2}{\Delta S^2} + r\right), \quad
  c_i = \frac{\Delta t}{4}\left(\frac{\sigma^2 S_i^2}{\Delta S^2} + \frac{r S_i}{\Delta S}\right)$$

The tridiagonal system and time-stepping logic are otherwise identical to the standard CN scheme.

In [ ]:
def solve_down_and_out_call(H, K, r, sigma, T, S_max, M, N):
    """
    Crank-Nicolson pricer for a down-and-out European call.

    Grid: S in [H, S_max] with M+1 nodes.
    Lower BC: V(H, t) = 0  (knock-out at barrier)
    Upper BC: V(S_max, t) = S_max - K * exp(-r*(T-t))  (deep ITM call)
    """
    dS = (S_max - H) / M
    dt = T / N
    S  = np.linspace(H, S_max, M + 1)

    V = np.maximum(S - K, 0.0)  # terminal payoff

    Si = S[1:M]  # interior node prices

    # CN coefficients using actual stock prices (not indices)
    a = 0.25 * dt * (sigma**2 * Si**2 / dS**2 - r * Si / dS)
    b = -0.5  * dt * (sigma**2 * Si**2 / dS**2 + r)
    c = 0.25  * dt * (sigma**2 * Si**2 / dS**2 + r * Si / dS)

    # Banded matrix for the implicit part: (I - 0.5*dt*L)
    ab = np.zeros((3, M - 1))
    ab[0, 1:]  = -c[:-1]   # superdiagonal
    ab[1, :]   = 1.0 - b   # main diagonal
    ab[2, :-1] = -a[1:]    # subdiagonal

    for n in range(N - 1, -1, -1):
        t       = n * dt
        tau     = T - t                                         # time to expiry
        V_lower = 0.0                                           # knock-out BC
        V_upper = max(S_max - K * np.exp(-r * tau), 0.0)       # call upper BC

        rhs        = a * V[0:M-1] + (1.0 + b) * V[1:M] + c * V[2:M+1]
        rhs[0]    += a[0]  * V_lower   # implicit lower boundary
        rhs[-1]   += c[-1] * V_upper   # implicit upper boundary

        V[1:M] = solve_banded((1, 1), ab, rhs)
        V[0]   = V_lower
        V[M]   = V_upper

    return S, V

## 3. Validation Against Merton

In [ ]:
S_fd, V_do = solve_down_and_out_call(H, K, r, sigma, T, S_max, M, N)

fd_price = float(np.interp(S0, S_fd, V_do))

print(f'Down-and-out call (FD, M=N={M}): {fd_price:.4f}')
print(f'Down-and-out call (Merton):      {analytical:.4f}')
print(f'Absolute error:                  {abs(fd_price - analytical):.4f}')

assert abs(fd_price - analytical) < 0.05, 'FD price deviates too far from Merton'

## 4. In–Out Parity and the Knock-In Call

A down-and-in call pays off exactly when the down-and-out call does not. Together they replicate the vanilla call:

$$C_{\text{do}} + C_{\text{di}} = C_{\text{vanilla}}$$

This **in–out parity** holds model-independently for European barriers. It lets us price the knock-in call for free once we have the knock-out price, without solving a separate PDE.

In [ ]:
_, V_vanilla = solve_crank_nicolson(S_max, K, r, sigma, T, M, N, option_type='call')

# vanilla grid spans [0, S_max]; barrier grid spans [H, S_max]
# interpolate vanilla onto the barrier grid for a fair comparison
V_van_interp = np.interp(S_fd, np.linspace(0, S_max, M + 1), V_vanilla)

# knock-in via parity: C_di = C_vanilla - C_do
V_di = V_van_interp - V_do

di_price = float(np.interp(S0, S_fd, V_di))
do_price = float(np.interp(S0, S_fd, V_do))
parity   = do_price + di_price

print(f'Knock-out call (FD):    {do_price:.4f}')
print(f'Knock-in  call (FD):    {di_price:.4f}')
print(f'Sum (should = vanilla): {parity:.4f}')
print(f'Vanilla (FD):           {float(np.interp(S0, np.linspace(0, S_max, M+1), V_vanilla)):.4f}')

## 5. Price Profiles

In [ ]:
mask    = S_fd <= 200
S_plot  = S_fd[mask]

# analytical curves for comparison
C_van_curve = np.array([bs_call(s, K, T, r, sigma) for s in S_plot])
C_do_curve  = np.array([down_and_out_call_analytical(s, K, T, r, sigma, H)
                        if s > H else 0.0 for s in S_plot])
C_di_curve  = C_van_curve - C_do_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: knock-out vs vanilla
ax = axes[0]
ax.plot(S_plot, C_van_curve,         label='Vanilla call')
ax.plot(S_plot, C_do_curve, '--',    label=f'Down-and-out (H={H})')
ax.plot(S_plot, np.maximum(S_plot - K, 0), ':', color='gray', label='Intrinsic')
ax.axvline(H, color='r', lw=0.8, ls='--', alpha=0.6, label=f'Barrier H={H}')
ax.axvline(K, color='k', lw=0.8, ls='--', alpha=0.3, label=f'Strike K={K}')
ax.set_xlabel('Stock price S')
ax.set_ylabel('Option value')
ax.set_title('Down-and-Out Call vs Vanilla')
ax.legend(fontsize=8)
ax.grid(True, ls='--', alpha=0.4)

# Right: knock-in
ax = axes[1]
ax.plot(S_plot, C_di_curve,          label=f'Down-and-in  (H={H})')
ax.plot(S_plot, C_van_curve, '--',   label='Vanilla call', alpha=0.5)
ax.axvline(H, color='r', lw=0.8, ls='--', alpha=0.6, label=f'Barrier H={H}')
ax.axvline(K, color='k', lw=0.8, ls='--', alpha=0.3, label=f'Strike K={K}')
ax.set_xlabel('Stock price S')
ax.set_ylabel('Option value')
ax.set_title('Down-and-In Call via In–Out Parity')
ax.legend(fontsize=8)
ax.grid(True, ls='--', alpha=0.4)

plt.tight_layout()
plt.savefig('../results/09_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

The down-and-out call tracks the vanilla call for $S \gg H$ but drops sharply as $S$ approaches $H$, reaching zero at the barrier. The down-and-in call shows the mirror behavior: it is nearly worthless far above $H$ (the path is unlikely to ever touch the barrier) but approaches the vanilla price as $S \to H$ (the stock is almost certain to touch the barrier).

## 6. Barrier Level Sensitivity

How does the knock-out price vary with the barrier level? A higher barrier is more likely to be hit, reducing the knock-out value and increasing the knock-in value.

In [ ]:
barriers    = np.linspace(50, 98, 30)
do_prices   = []
di_prices   = []

for h in barriers:
    c_do = down_and_out_call_analytical(S0, K, T, r, sigma, h)
    c_di = vanilla_call - c_do
    do_prices.append(c_do)
    di_prices.append(c_di)

plt.figure(figsize=(8, 4))
plt.plot(barriers, do_prices, label='Down-and-out call')
plt.plot(barriers, di_prices, label='Down-and-in  call')
plt.axhline(vanilla_call, color='gray', ls=':', lw=1.2, label=f'Vanilla call ({vanilla_call:.2f})')
plt.xlabel('Barrier level H')
plt.ylabel('Option value')
plt.title(f'Barrier Option Value vs Barrier Level  (S={S0}, K={K}, T={T})')
plt.legend()
plt.grid(True, ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../results/09_barrier_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nAt H=50 (far barrier): C_do={do_prices[0]:.4f}, C_di={di_prices[0]:.4f}')
print(f'At H=95 (near barrier): C_do={do_prices[-1]:.4f}, C_di={di_prices[-1]:.4f}')

## 7. Convergence

We refine the grid and compare FD prices against the Merton formula. Because our grid always includes $H$ as a node (via `np.linspace(H, S_max, M+1)`), the barrier is exactly resolved at every resolution — there is no barrier-oscillation error and convergence is monotone.

In [ ]:
ref = down_and_out_call_analytical(S0, K, T, r, sigma, H)
print(f'Merton reference: {ref:.6f}')

grid_sizes = [20, 40, 80, 120, 160, 200, 300, 400]
errors     = []

for g in grid_sizes:
    S_g, V_g = solve_down_and_out_call(H, K, r, sigma, T, S_max, g, g)
    price_g  = float(np.interp(S0, S_g, V_g))
    err      = abs(price_g - ref)
    errors.append(err)
    print(f'M=N={g:4d}  price={price_g:.6f}  error={err:.6f}')

# fit convergence slope on last few points
log_h  = np.log(1.0 / np.array(grid_sizes))
log_e  = np.log(errors)
slope  = np.polyfit(log_h[-5:], log_e[-5:], 1)[0]
print(f'\nEstimated convergence order: {slope:.2f}')

plt.figure(figsize=(7, 4))
plt.loglog(grid_sizes, errors, 'o-', label='FD error vs Merton')
# reference O(h^2) line
h_ref = np.array(grid_sizes, dtype=float)
plt.loglog(h_ref, errors[0] * (h_ref[0] / h_ref)**2, '--', color='gray',
           alpha=0.7, label=r'$O(M^{-2})$ reference')
plt.xlabel('Grid size M (= N)')
plt.ylabel('Absolute error vs Merton')
plt.title('Down-and-Out Call Convergence (Crank–Nicolson)')
plt.legend()
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../results/09_convergence.png', dpi=150, bbox_inches='tight')
plt.show()